# Data Transformation

In [1]:
import pandas as pd

In [2]:
form_window = 4
class_window = 15

In [3]:
def date_transform(date):
    date = date.replace("Jan", "1")
    date = date.replace("Feb", "2")
    date = date.replace("Mar", "3")
    date = date.replace("Apr", "4")
    date = date.replace("May", "5")
    date = date.replace("Jun", "6")
    date = date.replace("Jul", "7")
    date = date.replace("Aug", "8")
    date = date.replace("Sep", "9")
    date = date.replace("Oct", "10")
    date = date.replace("Nov", "11")
    date = date.replace("Dec", "12")
    date = date.replace("th", "")
    date = date.replace("st", "")
    date = date.replace("nd", "")
    date = date.replace("rd", "")
    return date

def clean_sheet(goals_c):
    if goals_c == 0:
        return 1
    else:
        return 0

In [4]:
team_stats_df = pd.read_csv('../data/master/master_team_stats.csv')
team_stats_df['date'] = team_stats_df['date'].apply(date_transform)
team_stats_df['date'] = pd.to_datetime(team_stats_df['date'], format='%d %m %Y')
team_stats_df = team_stats_df.sort_values(by=['date', 'match_id'])
team_stats_df['cs'] = 0
team_stats_df['cs'] = team_stats_df['goals_c'].apply(clean_sheet)
team_stats_df['data_point'] = True
team_stats_df


,match_id,date,team,goals,xg,goals_c,xg_c,cs,data_point
0,918893,2017-08-11,ARS,4,2.58,3,1.50,0,True
1,918893,2017-08-11,LEI,3,1.50,4,2.58,0,True
2,918894,2017-08-12,BHA,0,0.29,2,1.93,0,True
3,918894,2017-08-12,MCI,2,1.93,0,0.29,1,True
4,918895,2017-08-12,BUR,3,0.65,2,1.53,0,True
...,...,...,...,...,...,...,...,...,...
6075,2444847,2025-05-25,SOU,1,0.62,2,2.39,0,True
6076,2444848,2025-05-25,BHA,4,2.24,1,2.03,0,True
6077,2444848,2025-05-25,TOT,1,2.03,4,2.24,0,True
6078,2444849,2025-05-25,BRE,1,1.43,1,1.04,0,True


In [5]:
print(team_stats_df[team_stats_df['match_id'] == 919270])

     match_id       date team  goals    xg  goals_c  xg_c  cs  data_point
754    919270 2018-05-13  STK      2  1.64        1  3.00   0        True
755    919270 2018-05-13  SWA      1  3.00        2  1.64   0        True


In [6]:
### Check that dates are in correct range i.e check whats before august 2017 and after ... 2025

In [7]:
season_changes = {
    "2018-08-01": {
        "promoted": ["WOL", "CAR", "FUL"],
        "relegated": ["SWA", "STK", "WBA"]
    },
    "2019-08-01": {
        "promoted": ["NOR", "SHU", "AVL"],
        "relegated": ["CAR", "FUL", "HUD"]
    },
    "2020-08-01": {
        "promoted": ["LEE", "WBA", "FUL"],
        "relegated": ["BOU", "WAT", "NOR"]
    },
    "2021-08-01": {
        "promoted": ["NOR", "WAT", "BRE"],
        "relegated": ["FUL", "WBA", "SHU"]
    },
    "2022-08-01": {
        "promoted": ["FUL", "BOU", "NFO"],
        "relegated": ["BUR", "WAT", "NOR"]
    },
    "2023-08-01": {
        "promoted": ["BUR", "SHE", "LUT"],
        "relegated": ["LEI", "LEE", "SOU"]
    },
    "2024-08-01": {
        "promoted": ["LEI", "IPS", "SOU"],
        "relegated": ["LUT", "BUR", "SHU"]
    },
    "2025-08-01": {
        "promoted": ["LEE", "BUR", "SUN"],
        "relegated": ["LEI", "IPS", "SOU"]
    }
}


In [8]:
new_teams_df = pd.DataFrame()

for season_start, changes in season_changes.items():
    promoted_teams = changes['promoted']
    relegated_teams = changes['relegated']

    season_start_date = pd.to_datetime(season_start)

    # For each promoted team, get relegated team at same position
    for pos, promoted_team in enumerate(promoted_teams):
        corresponding_relegated = relegated_teams[pos]

        # Filter relegated team's games before season start
        lookalike_rows = team_stats_df[
            (team_stats_df['team'] == corresponding_relegated) &
            (team_stats_df['date'] < season_start_date)
        ]

        # Take last 'history_length' games
        selected_rows = lookalike_rows.tail(form_window+class_window).copy()

        # Change team and flag as synthetic data
        selected_rows['team'] = promoted_team
        selected_rows['data_point'] = False

        # Append to new_teams_df
        new_teams_df = pd.concat([new_teams_df, selected_rows], ignore_index=True)

# Finally add these synthetic rows to your main dataframe
team_stats_df = pd.concat([team_stats_df, new_teams_df], ignore_index=True)

# Sort by date and match_id if needed
team_stats_df = team_stats_df.sort_values(by=['date', 'match_id']).reset_index(drop=True)

team_stats_df


,match_id,date,team,goals,xg,goals_c,xg_c,cs,data_point
0,918893,2017-08-11,ARS,4,2.58,3,1.50,0,True
1,918893,2017-08-11,LEI,3,1.50,4,2.58,0,True
2,918894,2017-08-12,BHA,0,0.29,2,1.93,0,True
3,918894,2017-08-12,MCI,2,1.93,0,0.29,1,True
4,918895,2017-08-12,BUR,3,0.65,2,1.53,0,True
...,...,...,...,...,...,...,...,...,...
6531,2444847,2025-05-25,SUN,1,0.62,2,2.39,0,False
6532,2444848,2025-05-25,BHA,4,2.24,1,2.03,0,True
6533,2444848,2025-05-25,TOT,1,2.03,4,2.24,0,True
6534,2444849,2025-05-25,BRE,1,1.43,1,1.04,0,True


In [9]:
team_stats_df.tail(50)

,match_id,date,team,goals,xg,goals_c,xg_c,cs,data_point
6486,2444826,2025-05-11,NFO,2,1.30,2,1.05,0,True
6487,2444826,2025-05-11,LEE,2,1.05,2,1.30,0,False
6488,2444828,2025-05-11,CRY,2,3.36,0,0.73,1,True
6489,2444828,2025-05-11,TOT,0,0.73,2,3.36,0,True
6490,2444831,2025-05-16,AVL,2,1.39,0,0.51,1,True
6491,2444831,2025-05-16,TOT,0,0.51,2,1.39,0,True
6492,2444834,2025-05-16,CHE,1,0.82,0,0.29,1,True
6493,2444834,2025-05-16,MUN,0,0.29,1,0.82,0,True
6494,2444830,2025-05-18,ARS,1,0.64,0,1.86,1,True
6495,2444830,2025-05-18,NEW,0,1.86,1,0.64,0,True


In [10]:
team_stats_df.head(10)

,match_id,date,team,goals,xg,goals_c,xg_c,cs,data_point
0,918893,2017-08-11,ARS,4,2.58,3,1.50,0,True
1,918893,2017-08-11,LEI,3,1.50,4,2.58,0,True
2,918894,2017-08-12,BHA,0,0.29,2,1.93,0,True
3,918894,2017-08-12,MCI,2,1.93,0,0.29,1,True
4,918895,2017-08-12,BUR,3,0.65,2,1.53,0,True
5,918895,2017-08-12,CHE,2,1.53,3,0.65,0,True
6,918896,2017-08-12,CRY,0,1.07,3,1.50,0,True
7,918896,2017-08-12,HUD,3,1.50,0,1.07,1,True
8,918897,2017-08-12,EVE,1,0.60,0,0.37,1,True
9,918897,2017-08-12,STK,0,0.37,1,0.60,0,True


In [11]:
team_stats_df[team_stats_df['data_point'] == False]

,match_id,date,team,goals,xg,goals_c,xg_c,cs,data_point
386,919086,2017-12-26,CAR,1,1.75,1,1.91,0,False
389,919087,2017-12-26,WOL,0,0.39,5,3.09,0,False
398,919092,2017-12-26,FUL,0,1.47,0,0.46,1,False
407,919094,2017-12-30,CAR,0,0.25,5,2.63,0,False
418,919101,2017-12-30,WOL,2,1.15,1,1.23,0,False
...,...,...,...,...,...,...,...,...,...
6503,2444837,2025-05-18,LEE,2,0.80,0,1.41,1,False
6504,2444837,2025-05-18,BUR,0,1.41,2,0.80,0,False
6515,2444840,2025-05-25,LEE,0,0.27,2,1.62,0,False
6520,2444842,2025-05-25,BUR,1,0.74,3,1.08,0,False


In [12]:
match_id_date_map = {}
match_id_teams_map = {}
for index, row in team_stats_df.iterrows():
    if row['data_point'] == True:
        if row['match_id'] not in match_id_date_map:
            match_id_date_map[row['match_id']] = row['date']
        if row['match_id'] not in match_id_teams_map:
            match_id_teams_map[row['match_id']] = []
        match_id_teams_map[row['match_id']].append(row['team'])

If class_window is 0, then it sums stats from the most recent form_window games before the given date.

If class_window is greater than 0, it skips the most recent form_window games (by decreasing skip_count) and then sums stats for the next class_window games going backward in time.

In other words:

When class_window > 0, the function ignores the most recent form_window matches and only accumulates stats for the class_window matches before those.

if you specify a nonzero class_window, only those class_window matches are used for the stats, and the form_window matches are skipped (not counted).

In [13]:
def getTeamFeatures(team, date, form_window, class_window=0):
    """
    Get features for a team at a specific date.
    """
    team_features = {}
    team_features['team_name'] = team
    team_features['date'] = date
    team_features['form'] = form_window
    team_features['class'] = class_window

    xg = 0
    xg_c = 0
    goals = 0
    goals_c = 0
    cs = 0
    lookback_count = 0
    if class_window == 0:
        skip_count = 0
    else:
        skip_count = form_window
    for index, row in team_stats_df.loc[(team_stats_df['team'] == team)].sort_values(by=['date'], ascending=False).iterrows():        
        if row['date'] < pd.to_datetime(date):
            if skip_count == 0:
                xg = xg + row['xg']
                xg_c = xg_c + row['xg_c']
                goals = goals + row['goals']
                goals_c = goals_c + row['goals_c']
                cs = cs + row['cs']
                lookback_count += 1
            else:
                skip_count -= 1
        if lookback_count == max(form_window, class_window):
            break
    team_features['xg'] = xg
    team_features['xg_c'] = xg_c
    team_features['goals'] = goals
    team_features['goals_c'] = goals_c
    team_features['cs'] = cs
    team_features['lookback_count'] = lookback_count

    return team_features

In [14]:
print(team_stats_df.columns.tolist())

['match_id', 'date', 'team', 'goals', 'xg', 'goals_c', 'xg_c', 'cs', 'data_point']


In [15]:
print(getTeamFeatures('WOL', '05-14-2018', 4))
print(getTeamFeatures('CRY', '09-23-2017', 4, 15))

{'team_name': 'WOL', 'date': '05-14-2018', 'form': 4, 'class': 0, 'xg': 5.529999999999999, 'xg_c': 6.29, 'goals': 1, 'goals_c': 5, 'cs': 0, 'lookback_count': 4}
{'team_name': 'CRY', 'date': '09-23-2017', 'form': 4, 'class': 15, 'xg': 1.07, 'xg_c': 1.5, 'goals': 0, 'goals_c': 3, 'cs': 0, 'lookback_count': 1}


In [16]:
print(team_stats_df[team_stats_df['team'] == 'WOL'])


      match_id       date team  goals    xg  goals_c  xg_c  cs  data_point
389     919087 2017-12-26  WOL      0  0.39        5  3.09   0       False
418     919101 2017-12-30  WOL      2  1.15        1  1.23   0       False
441     919111 2018-01-02  WOL      0  0.44        2  1.35   0       False
457     919119 2018-01-13  WOL      1  1.33        1  1.45   0       False
494     919131 2018-01-22  WOL      1  0.16        0  2.52   1       False
...        ...        ...  ...    ...   ...      ...   ...  ..         ...
6436   2444809 2025-04-26  WOL      3  2.02        0  1.15   1        True
6445   2444818 2025-05-02  WOL      0  0.40        1  0.71   0        True
6478   2444829 2025-05-10  WOL      0  0.91        2  1.57   0        True
6510   2444835 2025-05-20  WOL      2  1.41        4  1.66   0        True
6535   2444849 2025-05-25  WOL      1  1.04        1  1.43   0        True

[285 rows x 9 columns]


In [17]:
print(getTeamFeatures('LIV', '06-23-2024', 4, 15))

{'team_name': 'LIV', 'date': '06-23-2024', 'form': 4, 'class': 15, 'xg': 39.94, 'xg_c': 18.3, 'goals': 36, 'goals_c': 18, 'cs': 2, 'lookback_count': 15}


# Do the same for player data

In [18]:
player_stats_bonus_df = pd.read_csv('../data/master/master_bonus.csv')
player_stats_defending_df = pd.read_csv('../data/master/master_defending.csv')
player_stats_expected_df = pd.read_csv('../data/master/master_expected.csv')
player_stats_fantasy_df = pd.read_csv('../data/master/master_fantasy.csv')
player_stats_keeping_df = pd.read_csv('../data/master/master_keeping.csv')
player_stats_pen_misses_df = pd.read_csv('../data/master/master_pen_misses.csv')

player_stats_df = pd.concat([player_stats_bonus_df,
                            player_stats_defending_df.drop(['match_id', 'nick_name', 'full_name', 'pos', 'team'], axis=1),
                            player_stats_expected_df.drop(['match_id', 'nick_name', 'full_name', 'pos', 'team'], axis=1),
                            player_stats_fantasy_df.drop(['match_id', 'nick_name', 'full_name', 'pos', 'team'], axis=1),
                            player_stats_pen_misses_df.drop(['match_id', 'nick_name', 'full_name', 'pos', 'team'], axis=1)], axis=1)

player_stats_df = pd.merge(player_stats_df, player_stats_keeping_df.drop(['full_name', 'pos', 'team'], axis=1), on=['match_id', 'nick_name'], how='left')
player_stats_df['date'] = ""
player_stats_df['bonus'] = 0
player_stats_df['points'] = 0


In [19]:
player_stats_df

,match_id,nick_name,full_name,pos,team,bps,clearences,blocks,interceptions,recoveries,...,goals_c,own_goals,y_c,r_c,bonus,points,pen_miss,saves,pen_saves,date
0,918893,Cech,Petr Cech,Goalkeeper,ARS,9,0,0,0,4,...,3,0,0,0,0,0,0,1.0,0.0,
1,918893,Walcott,Theo Walcott,Midfielder,ARS,2,0,0,1,0,...,0,0,0,0,0,0,0,NaN,NaN,
2,918893,Ozil,Mesut Ozil,Midfielder,ARS,9,1,0,0,2,...,3,0,0,0,0,0,0,NaN,NaN,
3,918893,Monreal,Nacho Monreal,Defender,ARS,24,7,0,5,7,...,3,0,0,0,0,0,0,NaN,NaN,
4,918893,Ramsey,Aaron Ramsey,Midfielder,ARS,15,0,0,0,2,...,0,0,0,0,0,0,0,NaN,NaN,
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
86711,2444849,Ait Nouri,Rayan Ait Nouri,Defender,WOL,15,3,0,1,3,...,1,0,0,0,0,0,0,NaN,NaN,
86712,2444849,Rodrigo Gomes,Rodrigo Martins Gomes,Midfielder,WOL,4,0,0,2,0,...,0,0,0,0,0,0,0,NaN,NaN,
86713,2444849,André,André Trindade da Costa Neto,Midfielder,WOL,15,0,0,0,4,...,1,0,0,0,0,0,0,NaN,NaN,
86714,2444849,Toti,Toti António Gomes,Defender,WOL,8,3,0,1,4,...,1,0,0,0,0,0,0,NaN,NaN,


In [20]:
player_stats_df.loc[6500]

match_id                  919129
nick_name        Stephens (Jack)
full_name          Jack Stephens
pos                     Defender
team                         SOU
bps                            9
clearences                     6
blocks                         1
interceptions                  3
recoveries                     7
tackles_won                    3
xa                           0.0
xg                          0.12
cost                         4.0
mins                          90
goals                          0
assists                        0
cs                             0
goals_c                        1
own_goals                      0
y_c                            1
r_c                            0
bonus                          0
points                         0
pen_miss                       0
saves                        NaN
pen_saves                    NaN
date                            
Name: 6500, dtype: object

# Calculate bonus points from bps

In [21]:
for match in player_stats_df.match_id.unique():
    match_players = player_stats_df.loc[player_stats_df['match_id'] == match]
    bps_groups = match_players.groupby('bps')

    bonus_assigned = 0

    # Iterate groups sorted by bps descending
    for bps_value, group in sorted(bps_groups, key=lambda x: x[0], reverse=True):
        size = len(group)
        if bonus_assigned == 0:
            points = [3] * size
            bonus_assigned += size
        elif bonus_assigned == 1:
            points = [2] * size
            bonus_assigned += size
        elif bonus_assigned == 2:
            points = [1] * size
            bonus_assigned += size
        else:
            points = [0] * size
        
        for idx, point in zip(group.index, points):
            player_stats_df.at[idx, 'bonus'] = point


In [22]:
player_stats_df[player_stats_df['match_id'] == 918993][['nick_name', 'bps']]

,nick_name,bps
2757,Fabregas,14
2758,Cahill,20
2759,Azpilicueta,42
2760,Hazard,3
2761,Willian,3
2762,Courtois,24
2763,Drinkwater,3
2764,Alonso,21
2765,Morata,29
2766,Rüdiger,4


In [23]:
player_stats_df[player_stats_df['match_id'] == 918993]

,match_id,nick_name,full_name,pos,team,bps,clearences,blocks,interceptions,recoveries,...,goals_c,own_goals,y_c,r_c,bonus,points,pen_miss,saves,pen_saves,date
2757,918993,Fabregas,Francesc Fábregas,Midfielder,CHE,14,1,1,2,5,...,0,0,0,0,0,0,0,NaN,NaN,
2758,918993,Cahill,Gary Cahill,Defender,CHE,20,3,0,1,1,...,0,0,0,0,0,0,0,NaN,NaN,
2759,918993,Azpilicueta,César Azpilicueta,Defender,CHE,42,1,0,3,6,...,0,0,0,0,3,0,0,NaN,NaN,
2760,918993,Hazard,Eden Hazard,Midfielder,CHE,3,0,0,0,3,...,0,0,0,0,0,0,0,NaN,NaN,
2761,918993,Willian,Willian Borges Da Silva,Midfielder,CHE,3,0,0,0,1,...,0,0,0,0,0,0,0,NaN,NaN,
2762,918993,Courtois,Thibaut Courtois,Goalkeeper,CHE,24,0,0,0,6,...,0,0,0,0,0,0,0,3.0,0.0,
2763,918993,Drinkwater,Daniel Drinkwater,Midfielder,CHE,3,1,0,0,1,...,0,0,0,0,0,0,0,NaN,NaN,
2764,918993,Alonso,Marcos Alonso,Defender,CHE,21,1,0,2,8,...,0,0,0,0,0,0,0,NaN,NaN,
2765,918993,Morata,Álvaro Morata,Forward,CHE,29,1,1,0,6,...,0,0,0,0,2,0,0,NaN,NaN,
2766,918993,Rüdiger,Antonio Rüdiger,Defender,CHE,4,2,0,0,2,...,0,0,0,0,0,0,0,NaN,NaN,


In [24]:
import pandas as pd

pd.set_option('display.max_columns', None)  # show all columns
pd.set_option('display.width', None)        # don't wrap lines


In [25]:
import pandas as pd
pen_miss_df = pd.read_csv('../data/master/master_pen_misses.csv')
pen_miss_df[pen_miss_df['pen_miss'] == 0]

,match_id,nick_name,full_name,pos,team,pen_miss
0,918893,Cech,Petr Cech,Goalkeeper,ARS,0
1,918893,Walcott,Theo Walcott,Midfielder,ARS,0
2,918893,Ozil,Mesut Ozil,Midfielder,ARS,0
3,918893,Monreal,Nacho Monreal,Defender,ARS,0
4,918893,Ramsey,Aaron Ramsey,Midfielder,ARS,0
...,...,...,...,...,...,...
86711,2444849,Ait Nouri,Rayan Ait Nouri,Defender,WOL,0
86712,2444849,Rodrigo Gomes,Rodrigo Martins Gomes,Midfielder,WOL,0
86713,2444849,André,André Trindade da Costa Neto,Midfielder,WOL,0
86714,2444849,Toti,Toti António Gomes,Defender,WOL,0


In [26]:
for index, row in player_stats_df.iterrows():
    player_stats_df.at[index, 'date'] = pd.to_datetime(match_id_date_map[row['match_id']])
    player_stats_df.at[index, 'points'] = 0
    player_stats_df.at[index, 'points'] += row['assists'] * 3
    player_stats_df.at[index, 'points'] -= row['y_c']
    player_stats_df.at[index, 'points'] -= row['r_c'] * 3
    player_stats_df.at[index, 'points'] -= row['own_goals'] * 2
    player_stats_df.at[index, 'points'] += row['bonus']
    player_stats_df.at[index, 'points'] -= row['pen_miss'] / -3

    defending_def_points = row['clearences'] + row['blocks'] + row['interceptions'] + row['tackles_won']
    defending_mid_att_points = defending_def_points + row['recoveries']
    
    if row['pos'] == "Goalkeeper":
        player_stats_df.at[index, 'points'] += int(row['saves']/3)
        player_stats_df.at[index, 'points'] += row['pen_saves']*5
        player_stats_df.at[index, 'points'] += row['goals'] * 6
        player_stats_df.at[index, 'points'] += row['cs'] * 4
        player_stats_df.at[index, 'points'] -= int(row['goals_c'] /2)
    elif row['pos'] == "Defender":
        player_stats_df.at[index, 'points'] += row['goals'] * 6
        player_stats_df.at[index, 'points'] += row['cs'] * 4
        player_stats_df.at[index, 'points'] -= int(row['goals_c'] /2)
        if defending_def_points >= 10:
            player_stats_df.at[index, 'points'] += 2
    elif row['pos'] == "Midfielder":
        player_stats_df.at[index, 'points'] += row['goals'] * 5
        player_stats_df.at[index, 'points'] += row['cs']
        if defending_mid_att_points >= 12:
            player_stats_df.at[index, 'points'] += 2
    else:  # Forward
        player_stats_df.at[index, 'points'] += row['goals'] * 4
        if defending_mid_att_points >= 12:
            player_stats_df.at[index, 'points'] += 2
    
    if row['mins'] > 0 and row['mins'] < 60:
        player_stats_df.at[index, 'points'] += 1
    elif row['mins'] >= 60:
        player_stats_df.at[index, 'points'] += 2
    
    if match_id_teams_map[row['match_id']][0] == row['team']:
        player_stats_df.at[index, 'opponent'] = match_id_teams_map[row['match_id']][1]
    else:
        player_stats_df.at[index, 'opponent'] = match_id_teams_map[row['match_id']][0]


player_stats_df = player_stats_df.sort_values(by=['date', 'match_id']).reset_index(drop=True)
player_stats_df['datapoint'] = True
player_stats_df


,match_id,nick_name,full_name,pos,team,bps,clearences,blocks,interceptions,recoveries,tackles_won,xa,xg,cost,mins,goals,assists,cs,goals_c,own_goals,y_c,r_c,bonus,points,pen_miss,saves,pen_saves,date,opponent,datapoint
0,918893,Cech,Petr Cech,Goalkeeper,ARS,9,0,0,0,4,1,0.00,0.00,0.0,90,0,0,0,3,0,0,0,0,1,0,1.0,0.0,2017-08-11 00:00:00,LEI,True
1,918893,Walcott,Theo Walcott,Midfielder,ARS,2,0,0,1,0,0,0.00,0.00,0.0,21,0,0,0,0,0,0,0,0,1,0,NaN,NaN,2017-08-11 00:00:00,LEI,True
2,918893,Ozil,Mesut Ozil,Midfielder,ARS,9,1,0,0,2,0,0.22,0.14,0.0,90,0,0,0,3,0,0,0,0,2,0,NaN,NaN,2017-08-11 00:00:00,LEI,True
3,918893,Monreal,Nacho Monreal,Defender,ARS,24,7,0,5,7,2,0.01,0.00,0.0,90,0,0,0,3,0,0,0,0,3,0,NaN,NaN,2017-08-11 00:00:00,LEI,True
4,918893,Ramsey,Aaron Ramsey,Midfielder,ARS,15,0,0,0,2,0,0.07,0.43,0.0,29,1,0,0,0,0,0,0,0,6,0,NaN,NaN,2017-08-11 00:00:00,LEI,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
86711,2444849,Ait Nouri,Rayan Ait Nouri,Defender,WOL,15,3,0,1,3,2,0.17,0.05,6.0,65,0,0,0,1,0,0,0,0,2,0,NaN,NaN,2025-05-25 00:00:00,BRE,True
86712,2444849,Rodrigo Gomes,Rodrigo Martins Gomes,Midfielder,WOL,4,0,0,2,0,0,0.01,0.00,4.5,30,0,0,0,0,0,0,0,0,1,0,NaN,NaN,2025-05-25 00:00:00,BRE,True
86713,2444849,André,André Trindade da Costa Neto,Midfielder,WOL,15,0,0,0,4,1,0.07,0.03,5.5,90,0,0,0,1,0,0,0,0,2,0,NaN,NaN,2025-05-25 00:00:00,BRE,True
86714,2444849,Toti,Toti António Gomes,Defender,WOL,8,3,0,1,4,1,0.01,0.00,4.5,90,0,0,0,1,0,0,0,0,2,0,NaN,NaN,2025-05-25 00:00:00,BRE,True


In [27]:
# Check minus
player_stats_df.loc[player_stats_df['points'] < -3][['match_id', 'nick_name', 'team', 'opponent', 'goals', 'points']]

,match_id,nick_name,team,opponent,goals,points
37168,2128500,Bednarek,SOU,MUN,0,-4
53438,2292845,Mepham,BOU,LIV,0,-4
71764,2367806,Bogle,SHU,ARS,0,-4
79250,2444603,Dawson,WOL,EVE,0,-4


In [28]:
player_stats_df.loc[player_stats_df['match_id'] == 918993]

,match_id,nick_name,full_name,pos,team,bps,clearences,blocks,interceptions,recoveries,tackles_won,xa,xg,cost,mins,goals,assists,cs,goals_c,own_goals,y_c,r_c,bonus,points,pen_miss,saves,pen_saves,date,opponent,datapoint
2922,918993,Fabregas,Francesc Fábregas,Midfielder,CHE,14,1,1,2,5,1,0.29,0.09,0.0,78,0,0,1,0,0,0,0,0,3,0,NaN,NaN,2017-11-05 00:00:00,MUN,True
2923,918993,Cahill,Gary Cahill,Defender,CHE,20,3,0,1,1,0,0.00,0.17,0.0,90,0,0,1,0,0,0,0,0,6,0,NaN,NaN,2017-11-05 00:00:00,MUN,True
2924,918993,Azpilicueta,César Azpilicueta,Defender,CHE,42,1,0,3,6,4,0.11,0.00,4.5,90,0,1,1,0,0,0,0,3,12,0,NaN,NaN,2017-11-05 00:00:00,MUN,True
2925,918993,Hazard,Eden Hazard,Midfielder,CHE,3,0,0,0,3,0,0.12,0.39,0.0,87,0,0,1,0,0,0,0,0,3,0,NaN,NaN,2017-11-05 00:00:00,MUN,True
2926,918993,Willian,Willian Borges Da Silva,Midfielder,CHE,3,0,0,0,1,0,0.13,0.00,5.0,8,0,0,0,0,0,0,0,0,1,0,NaN,NaN,2017-11-05 00:00:00,MUN,True
2927,918993,Courtois,Thibaut Courtois,Goalkeeper,CHE,24,0,0,0,6,0,0.00,0.00,5.5,90,0,0,1,0,0,0,0,0,7,0,3.0,0.0,2017-11-05 00:00:00,MUN,True
2928,918993,Drinkwater,Daniel Drinkwater,Midfielder,CHE,3,1,0,0,1,0,0.00,0.00,0.0,16,0,0,0,0,0,0,0,0,1,0,NaN,NaN,2017-11-05 00:00:00,MUN,True
2929,918993,Alonso,Marcos Alonso,Defender,CHE,21,1,0,2,8,1,0.04,0.02,0.0,90,0,0,1,0,0,0,0,0,6,0,NaN,NaN,2017-11-05 00:00:00,MUN,True
2930,918993,Morata,Álvaro Morata,Forward,CHE,29,1,1,0,6,0,0.02,0.30,8.4,90,1,0,1,0,0,0,0,2,8,0,NaN,NaN,2017-11-05 00:00:00,MUN,True
2931,918993,Rüdiger,Antonio Rüdiger,Defender,CHE,4,2,0,0,2,0,0.00,0.00,5.6,29,0,0,0,0,0,0,0,0,1,0,NaN,NaN,2017-11-05 00:00:00,MUN,True


In [29]:
player_stats_df.loc[player_stats_df['mins'] == 0]

,match_id,nick_name,full_name,pos,team,bps,clearences,blocks,interceptions,recoveries,tackles_won,xa,xg,cost,mins,goals,assists,cs,goals_c,own_goals,y_c,r_c,bonus,points,pen_miss,saves,pen_saves,date,opponent,datapoint
1641,918943,Maitland-Niles,Ainsley Maitland-Niles,Midfielder,ARS,3,0,0,0,0,0,0.00,0.00,0.0,0,0,0,0,0,0,0,0,0,0,0,NaN,NaN,2017-09-25 00:00:00,WBA,True
5801,919105,Matip,Joel Matip,Defender,LIV,3,0,0,0,0,0,0.00,0.00,4.8,0,0,0,0,0,0,0,0,0,0,0,NaN,NaN,2018-01-01 00:00:00,BUR,True
8066,919188,Darmian,Matteo Darmian,Defender,MUN,3,0,0,0,0,0,0.00,0.00,4.9,0,0,0,0,0,0,0,0,0,0,0,NaN,NaN,2018-03-10 00:00:00,LIV,True
10155,919202,Bailly,Eric Bailly,Defender,MUN,3,0,0,0,0,0,0.00,0.00,4.0,0,0,0,0,0,0,0,0,0,0,0,NaN,NaN,2018-05-10 00:00:00,WHU,True
19581,987930,Williams (Danny),Danny Williams,Midfielder,HUD,3,0,0,0,0,0,0.00,0.00,0.0,0,0,0,0,0,0,0,0,0,0,0,NaN,NaN,2019-04-13 00:00:00,TOT,True
20292,987951,Gibbs-White,Morgan Gibbs-White,Midfielder,WOL,3,0,0,0,0,0,0.00,0.00,7.5,0,0,0,0,0,0,0,0,0,0,0,NaN,NaN,2019-04-27 00:00:00,WAT,True
34584,2128405,Nakamba,Marvelous Nakamba,Midfielder,AVL,3,0,0,0,1,0,0.00,0.00,4.3,0,0,0,0,0,0,0,0,0,0,0,NaN,NaN,2020-12-12 00:00:00,WOL,True
39840,2128591,Shackleton,Jamie Shackleton,Midfielder,LEE,3,0,0,0,0,0,0.00,0.00,0.0,0,0,0,0,0,0,0,0,0,0,0,NaN,NaN,2021-04-10 00:00:00,MCI,True
44022,2210349,Tella,Nathan Tella,Midfielder,SOU,3,0,0,0,0,0,0.00,0.00,0.0,0,0,0,0,0,0,0,0,0,0,0,NaN,NaN,2021-10-16 00:00:00,LEE,True
56749,2292970,Elneny,Mohamed Elneny,Midfielder,ARS,3,0,0,0,0,0,0.00,0.00,4.4,0,0,0,0,0,0,0,0,0,0,0,NaN,NaN,2022-12-26 00:00:00,WHU,True


In [38]:
player_stats_df.loc[player_stats_df['full_name'] == "Roberto Firmino Barbosa de Oliveira"]

,match_id,nick_name,full_name,pos,team,bps,clearences,blocks,interceptions,recoveries,tackles_won,xa,xg,cost,mins,goals,assists,cs,goals_c,own_goals,y_c,r_c,bonus,points,pen_miss,saves,pen_saves,date,opponent,datapoint
171,918901,Firmino,Roberto Firmino Barbosa de Oliveira,Forward,LIV,40,1,0,0,3,0,0.10,0.84,0.0,80,1,1,0,2,0,0,0,3,12,0,NaN,NaN,2017-08-12 00:00:00,WAT,True
379,918907,Firmino,Roberto Firmino Barbosa de Oliveira,Forward,LIV,6,0,1,0,6,3,0.06,0.14,0.0,90,0,0,1,0,0,0,0,0,2,0,NaN,NaN,2017-08-19 00:00:00,CRY,True
767,918917,Firmino,Roberto Firmino Barbosa de Oliveira,Forward,LIV,46,0,0,0,3,0,0.31,0.36,0.0,80,1,1,1,0,0,0,0,3,12,0,NaN,NaN,2017-08-27 00:00:00,ARS,True
948,918928,Firmino,Roberto Firmino Barbosa de Oliveira,Forward,LIV,3,0,0,0,2,1,0.06,0.06,0.0,66,0,0,0,3,0,0,0,0,2,0,NaN,NaN,2017-09-09 00:00:00,MCI,True
1209,918937,Firmino,Roberto Firmino Barbosa de Oliveira,Forward,LIV,8,0,0,0,3,0,0.09,0.07,0.0,79,0,0,0,1,0,0,0,0,2,0,NaN,NaN,2017-09-16 00:00:00,BUR,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
60862,2292883,Firmino,Roberto Firmino Barbosa de Oliveira,Forward,LIV,13,0,0,0,6,0,0.06,0.03,0.0,65,0,0,1,0,0,0,0,0,2,0,NaN,NaN,2023-04-04 00:00:00,CHE,True
61225,2293105,Firmino,Roberto Firmino Barbosa de Oliveira,Forward,LIV,30,0,0,0,5,1,0.01,0.22,0.0,18,1,0,0,0,0,0,0,2,7,0,NaN,NaN,2023-04-09 00:00:00,ARS,True
61525,2293113,Firmino,Roberto Firmino Barbosa de Oliveira,Forward,LIV,5,0,0,0,0,0,0.00,0.17,0.0,13,0,0,0,0,0,0,0,0,1,0,NaN,NaN,2023-04-17 00:00:00,LEE,True
63175,2293173,Firmino,Roberto Firmino Barbosa de Oliveira,Forward,LIV,28,0,0,0,1,0,0.01,0.35,0.0,29,1,0,0,0,0,0,0,1,6,0,NaN,NaN,2023-05-20 00:00:00,AVL,True


In [52]:
def getPlayerFeatures(player, date, form_window, class_window=0):
    player_features = {}
    player_features['player_name'] = player
    player_features['date'] = date
    player_features['form_window'] = form_window
    player_features['class_window'] = class_window
    mins = 0
    goals = 0
    assists = 0
    cs = 0
    goals_c = 0
    own_goals = 0
    y_c = 0
    r_c = 0
    xg = 0
    xa = 0
    bps = 0
    saves = 0
    pen_saves = 0
    pen_miss = 0
    bonus = 0
    points = 0
    lookback_count = 0
    if class_window == 0:
        skip_count = 0
    else:
        skip_count = form_window
    for index, row in player_stats_df.loc[(player_stats_df['full_name'] == player)].sort_values(by=['date'], ascending=False).iterrows():        
        if row['date'] < pd.to_datetime(date):
            if skip_count == 0:
                mins += row['mins']
                print(row['mins'])
                goals += row['goals']
                assists += row['assists']
                cs += row['cs']
                goals_c += row['goals_c']
                own_goals += row['own_goals']
                y_c += row['y_c']
                r_c += row['r_c']
                xg += row['xg']
                xa += row['xa']
                bps += row['bps']
                try:
                    saves += row['saves']
                    pen_saves += row['pen_saves']
                except:
                    pass
                pen_miss += row['pen_miss']/-3
                bonus += row['bonus']
                points += row['points']
                lookback_count += 1
            else:
                skip_count -= 1
        if lookback_count == max(form_window, class_window):
            break
    player_features['mins'] = mins
    player_features['goals'] = goals
    player_features['assists'] = assists
    player_features['cs'] = cs
    player_features['goals_c'] = goals_c
    player_features['own_goals'] = own_goals
    player_features['y_c'] = y_c
    player_features['r_c'] = r_c
    player_features['xg'] = xg
    player_features['xa'] = xa
    player_features['bps'] = bps
    player_features['saves'] = saves
    player_features['pen_saves'] = pen_saves
    player_features['pen_miss'] = pen_miss
    player_features['bonus'] = bonus
    player_features['points'] = points
    player_features['lookback_count'] = lookback_count
    return player_features

print(getPlayerFeatures("Roberto Firmino Barbosa de Oliveira", '06-23-2024', 4, 15))
                

65
24
28
13
5
23
34
15
75
73
90
75
79
69
55
{'player_name': 'Roberto Firmino Barbosa de Oliveira', 'date': '06-23-2024', 'form_window': 4, 'class_window': 15, 'mins': 723, 'goals': 3, 'assists': 1, 'cs': 3, 'goals_c': 8, 'own_goals': 0, 'y_c': 0, 'r_c': 0, 'xg': 2.15, 'xa': 0.61, 'bps': 193, 'saves': nan, 'pen_saves': nan, 'pen_miss': 0.0, 'bonus': 2, 'points': 39, 'lookback_count': 15}


In [53]:
print(getPlayerFeatures("Dominic Solanke", '10-01-2024', 4, 15))

34
90
90
90
84
90
90
90
89
83
89
90
89
90
90
{'player_name': 'Dominic Solanke', 'date': '10-01-2024', 'form_window': 4, 'class_window': 15, 'mins': 1278, 'goals': 6, 'assists': 1, 'cs': 5, 'goals_c': 20, 'own_goals': 0, 'y_c': 1, 'r_c': 0, 'xg': 7.26, 'xa': 0.74, 'bps': 244, 'saves': nan, 'pen_saves': nan, 'pen_miss': 2.0, 'bonus': 10, 'points': 63, 'lookback_count': 15}


In [54]:
print(getPlayerFeatures("Alexis Sánchez", '01-01-2021', 20))

54
33
12
51
76
50
24
67
28
33
27
21
78
15
30
62
83
60
40
90
{'player_name': 'Alexis Sánchez', 'date': '01-01-2021', 'form_window': 20, 'class_window': 0, 'mins': 934, 'goals': 1, 'assists': 5, 'cs': 2, 'goals_c': 10, 'own_goals': 0, 'y_c': 3, 'r_c': 0, 'xg': 1.49, 'xa': 2.6900000000000004, 'bps': 206, 'saves': nan, 'pen_saves': nan, 'pen_miss': 0.0, 'bonus': 1, 'points': 47, 'lookback_count': 20}
